## Load packages and the input file

In [1]:
import sys, os
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd
from tabulate import tabulate
import h5py
import awkward as ak
import math

## Read file 

In [2]:
read_file = True
if read_file:
    final_df = pd.read_csv('./uubar.csv')
    print(tabulate(final_df.head(), headers='keys', tablefmt='psql'))
    print(tabulate(final_df.tail(), headers='keys', tablefmt='psql'))
print(final_df.shape)

+----+-------------+-------------+-------------+-------------+-------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+-----------------+-----------------+-----------------+-----------------+-----------------+-------------+-------------+-------------+-------------+-------------+---------+-----------+-----------+-----------+------------+----------------+------------------+------------------+------------------+----------------+---------+-----------+-----------+-----------+------------+----------------+------------------+------------------+------------------+----------------+---------+------------+------------+------------+------------+---------+------------+------------+------------+------------+---------+-----------+-----------+-----------+------------+---------+-----------+-----------+-----------+------------+----------------+--------------+--------------------+------------+
|    

## Feature preprocessing

In [6]:
NDIM = len(final_df.keys()) - 1
#dataset = final_df.values

# Count NaNs in each column
df_nonan = final_df.copy()
df_nonan = df_nonan.dropna()
#print(df_nonan.isna().sum())
dataset = df_nonan.values
X = dataset[:,0:NDIM]
Y = dataset[:,NDIM]

from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=7)

# preprocessing: standard scalar
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

# Check for NaNs/Infs in the dataset
X_train = np.nan_to_num(X_train)
X_test = np.nan_to_num(X_test)

## Particle transformer

[1, 128, 512, 128]


In [ ]:
# Plain (no-interaction) Particle Transformer for tabular / constituent-like inputs
# - 1. Particle embedding (MLP: 128 -> 512 -> 128 with LayerNorm + GELU)
# - 2. Particle attention blocks (standard MHSA, no pair/interaction bias)
# - 3. Class attention blocks (CaiT-style: cls queries particles)
#
# Works with:
#   x_particles: (B, P, F)  where:
#       B = batch size
#       P = number of “particles/tokens” per event (can be 69 if you treat each feature as a token)
#       or (B, 69, 1) each feature is treated as a “particle”
#       F = features per particle/token
#   mask (optional): (B, P) with 1 for real tokens and 0 for padded tokens

import torch
import torch.nn as nn
from typing import Optional, Sequence


class ParticleEmbed(nn.Module):
    """
    Per-particle MLP embedding like ParT:
      Linear -> LN -> GELU  (x3) with widths [128, 512, embed_dim]
    """
    def __init__(self, input_dim: int, embed_dims: Sequence[int] = (128, 512, 128)):
        super().__init__()
        dims = [input_dim] + list(embed_dims)
        layers = []
        for i in range(len(dims) - 1):
            layers += [
                nn.Linear(dims[i], dims[i + 1]),
                nn.LayerNorm(dims[i + 1]),
                nn.GELU(),
            ]
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, P, F)  like (B, 69, 1) each feature is treated as a “particle” -- token dim = 1
        return self.net(x)  # (B, P, C)


class ParticleBlock(nn.Module):
    """
    Transformer block (pre-LN) for particle self-attention.
    No interaction embedding / no pair bias.
    PyTorch module (a reusable layer). It will contain:
    self-attention
    feed-forward network (FFN)
    residual connections
    normalization
    dropout
    """
    def __init__(
        self,
        embed_dim: int = 128,                # vector size for each token (C)
        num_heads: int = 8,                  # number of attention heads               
        ffn_ratio: int = 4,                  # hidden size in FFN = embed_dim * ffn_ratio
        dropout: float = 0.1,
        attn_dropout: float = 0.1,
        activation_dropout: float = 0.1,
    ):
        super().__init__()

        # Multi-head self-attention
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=attn_dropout,
            batch_first=True,  # (B, P, C)
        )
        self.drop1 = nn.Dropout(dropout)

        # Feed-forward network (MLP)
        self.norm2 = nn.LayerNorm(embed_dim)
        hidden = embed_dim * ffn_ratio
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, hidden),
            nn.GELU(),
            nn.Dropout(activation_dropout),
            nn.Linear(hidden, embed_dim),
        )
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, key_padding_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        # x: (B, P, C)
        # key_padding_mask: (B, P) with True for PAD positions (PyTorch convention)
        h = self.norm1(x)
        h, _ = self.attn(h, h, h, key_padding_mask=key_padding_mask, need_weights=False)    # h for query, key, value
        x = x + self.drop1(h)

        h = self.norm2(x)
        h = self.ffn(h)
        x = x + self.drop2(h)
        return x


class ClassAttentionBlock(nn.Module):
    """
    CaiT-style class attention:
      - cls token attends to [cls + particles]
      - particles are NOT updated in cls blocks (only cls updates)
    """
    def __init__(
        self,
        embed_dim: int = 128,
        num_heads: int = 8,
        ffn_ratio: int = 4,
        dropout: float = 0.0,        # ParT often uses 0 in cls blocks
        attn_dropout: float = 0.0,
        activation_dropout: float = 0.0,
    ):
        super().__init__()

        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=attn_dropout,
            batch_first=True,  # (B, T, C)
        )
        self.drop1 = nn.Dropout(dropout)

        self.norm2 = nn.LayerNorm(embed_dim)
        hidden = embed_dim * ffn_ratio
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, hidden),
            nn.GELU(),
            nn.Dropout(activation_dropout),
            nn.Linear(hidden, embed_dim),
        )
        self.drop2 = nn.Dropout(dropout)

    def forward(
        self,
        x_particles: torch.Tensor,          # (B, P, C)
        x_cls: torch.Tensor,                # (B, 1, C)
        key_padding_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        # Build tokens [cls, particles]
        u = torch.cat([x_cls, x_particles], dim=1)  # (B, 1+P, C)

        # Expand padding mask to include cls (cls is never padded)
        if key_padding_mask is not None:
            cls_pad = torch.zeros((key_padding_mask.size(0), 1), device=key_padding_mask.device, dtype=torch.bool)
            pad_u = torch.cat([cls_pad, key_padding_mask], dim=1)  # (B, 1+P)
        else:
            pad_u = None

        # Attention: query = cls only, key/value = [cls + particles]
        residual = x_cls
        q = self.norm1(x_cls)  # (B, 1, C)
        kv = self.norm1(u)     # (B, 1+P, C)
        h, _ = self.attn(q, kv, kv, key_padding_mask=pad_u, need_weights=False)
        x_cls = residual + self.drop1(h)

        # FFN on cls only
        residual = x_cls
        h = self.norm2(x_cls)
        h = self.ffn(h)
        x_cls = residual + self.drop2(h)
        return x_cls  # (B, 1, C)


class PlainParticleTransformer(nn.Module):
    """
    Full model:
      embed -> N particle blocks -> M class-attention blocks -> norm -> head
    """
    def __init__(
        self,
        input_dim: int,               # features per token/particle
        num_classes: int = 2,
        num_tokens: Optional[int] = None,  # not required; kept for clarity
        embed_dims=(128, 512, 128),
        embed_dim: int = 128,         # must match last of embed_dims
        num_heads: int = 8,
        num_layers: int = 8,
        num_cls_layers: int = 2,
        ffn_ratio: int = 4,
        dropout: float = 0.1,
        attn_dropout: float = 0.1,
        activation_dropout: float = 0.1,
        cls_dropout: float = 0.0,     # typical ParT default for cls blocks
        fc_hidden: Optional[int] = None,  # e.g., 256 if you want an extra FC layer
    ):
        super().__init__()

        assert embed_dim == embed_dims[-1], "embed_dim must equal embed_dims[-1]"

        self.embed = ParticleEmbed(input_dim=input_dim, embed_dims=embed_dims)

        self.blocks = nn.ModuleList([
            ParticleBlock(
                embed_dim=embed_dim,
                num_heads=num_heads,
                ffn_ratio=ffn_ratio,
                dropout=dropout,
                attn_dropout=attn_dropout,
                activation_dropout=activation_dropout,
            )
            for _ in range(num_layers)
        ])

        self.cls_blocks = nn.ModuleList([
            ClassAttentionBlock(
                embed_dim=embed_dim,
                num_heads=num_heads,
                ffn_ratio=ffn_ratio,
                dropout=cls_dropout,
                attn_dropout=cls_dropout,
                activation_dropout=cls_dropout,
            )
            for _ in range(num_cls_layers)
        ])

        self.norm = nn.LayerNorm(embed_dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        nn.init.trunc_normal_(self.cls_token, std=0.02)

        if fc_hidden is None:
            self.head = nn.Linear(embed_dim, num_classes)
        else:
            self.head = nn.Sequential(
                nn.Linear(embed_dim, fc_hidden),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(fc_hidden, num_classes),
            )

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        x: (B, P, F)
        mask (optional): (B, P) with 1 for real tokens and 0 for padded tokens
        """
        if mask is None:
            key_padding_mask = None
        else:
            # PyTorch MHA expects True where positions are padded
            key_padding_mask = ~mask.bool()  # (B, P)

        # Embed particles
        x = self.embed(x)  # (B, P, C)

        # Zero-out padded tokens (optional but often helps)
        if mask is not None:
            x = x.masked_fill((~mask.bool()).unsqueeze(-1), 0.0)

        # Particle (self) attention blocks
        for blk in self.blocks:
            x = blk(x, key_padding_mask=key_padding_mask)

        # Class attention blocks (only update cls token)
        B = x.size(0)
        cls = self.cls_token.expand(B, 1, -1)  # (B, 1, C)
        for cblk in self.cls_blocks:
            cls = cblk(x_particles=x, x_cls=cls, key_padding_mask=key_padding_mask)

        cls = self.norm(cls).squeeze(1)  # (B, C)
        logits = self.head(cls)          # (B, num_classes)
        return logits


# -------------------------------------------------------------------------
# How to use it for YOUR case (30k events, 69 features):
# Option A (recommended for your current CSV tabular setup):
#   Treat each scalar feature as a "token" => P=69, F=1
#
#   X: (B, 69)  ->  x = X.unsqueeze(-1) => (B, 69, 1)
#   model = PlainParticleTransformer(input_dim=1, num_classes=2, num_layers=8, num_cls_layers=2)
#
# Option B (if you actually have constituent structure):
#   x already is (B, P, F) and mask is (B, P)
# -------------------------------------------------------------------------

if __name__ == "__main__":
    # Example for tabular X with 69 features => tokens=69, per-token dim=1
    B = 16
    X = torch.randn(B, 69)                 # your standardized features
    x_tokens = X.unsqueeze(-1)             # (B, 69, 1)

    model = PlainParticleTransformer(
        input_dim=1,
        num_classes=2,
        num_layers=8,
        num_cls_layers=2,
        num_heads=8,
        embed_dims=(128, 512, 128),
        dropout=0.1,
    )

    logits = model(x_tokens)               # (B, 2)
    print(logits.shape)


In [ ]:
# Input embedding

import torch
import torch.nn as nn

class ParticleEmbedding(nn.Module):
    def __init__(self, input_dim=17, embed_dim=128):
        super(ParticleEmbedding, self).__init__()
        
        # The paper specifies a 3-layer MLP with 128, 512, 128 nodes 
        # and LayerNorm (LN) for normalization.
        self.embedding_mlp = nn.Sequential(
            # Layer 1: Project 17 features to 128
            nn.Linear(input_dim, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            
            # Layer 2: Upscale to 512
            nn.Linear(128, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            
            # Layer 3: Final projection to embedding dimension (128)
            nn.Linear(512, embed_dim),
            nn.LayerNorm(embed_dim)
        )

    def forward(self, x):
        """
        Args:
            x: Input tensor of shape (batch_size, num_particles, 17)
        Returns:
            Embedded tensor of shape (batch_size, num_particles, 128)
        """
        # x is treated as a "particle cloud"
        return self.embedding_mlp(x)

In [ ]:
import torch
import torch.nn as nn

class PlainParticleAttentionBlock(nn.Module):
    def __init__(self, embed_dim=128, num_heads=8, mlp_ratio=4, dropout=0.1):
        super(PlainParticleAttentionBlock, self).__init__()
        
        # 1. Multi-Head Attention (Plain version without interaction bias)
        # In the paper, each head has a dimension of 16 (8 heads * 16 = 128)
        self.attn = nn.MultiheadAttention(
            embed_dim=embed_dim, 
            num_heads=num_heads, 
            dropout=dropout, 
            batch_first=True
        )
        
        # 2. The MLP (Feed-Forward Network)
        # The paper uses a 2-layer MLP with an expansion factor (128 -> 512 -> 128)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * mlp_ratio),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * mlp_ratio, embed_dim),
            nn.Dropout(dropout)
        )
        
        # 3. Layer Normalization (NormFormer style)
        # We need four norms: before and after the Attention, and before and after the MLP
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.norm3 = nn.LayerNorm(embed_dim)
        self.norm4 = nn.LayerNorm(embed_dim)
        
        # 4. Scaling Factors (Gamma)
        # NormFormer uses learnable scaling factors initialized at a small value
        self.gamma1 = nn.Parameter(torch.ones(embed_dim) * 1e-4)
        self.gamma2 = nn.Parameter(torch.ones(embed_dim) * 1e-4)

    def forward(self, x):
        """
        x: Input tensor of shape (batch_size, num_particles, embed_dim)
        """
        
        # --- Stage 1: Multi-Head Attention ---
        # Pre-Norm
        x_norm = self.norm1(x)
        
        # Attention (Self-attention: Q, K, and V are all the same)
        # Note: In a plain transformer, we don't add the Interaction Matrix 'U' here.
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        
        # Post-Norm + Scale + Residual Connection
        x = x + self.gamma1 * self.norm2(attn_out)
        
        # --- Stage 2: Feed-Forward Network (MLP) ---
        # Pre-Norm
        x_norm = self.norm3(x)
        
        # MLP
        mlp_out = self.mlp(x_norm)
        
        # Post-Norm + Scale + Residual Connection
        x = x + self.gamma2 * self.norm4(mlp_out)
        
        return x

In [ ]:
import torch
import torch.nn as nn

class ClassAttentionBlock(nn.Module):
    def __init__(self, embed_dim=128, num_heads=8, mlp_ratio=4, dropout=0.1):
        super(ClassAttentionBlock, self).__init__()
        
        # 1. Multi-Head Attention (MHA)
        # Note: In class attention, we use standard MHA (no physics interaction bias U)
        self.attn = nn.MultiheadAttention(
            embed_dim=embed_dim, 
            num_heads=num_heads, 
            dropout=dropout, 
            batch_first=True
        )

        # 2. Feed-Forward Network (MLP)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * mlp_ratio),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * mlp_ratio, embed_dim),
            nn.Dropout(dropout)
        )
        
        # 3. NormFormer-style Layer Normalization
        # We need norms for the Query (class token), the Keys/Values (particles), 
        # and for the MLP stages.
        self.norm_q = nn.LayerNorm(embed_dim)
        self.norm_k = nn.LayerNorm(embed_dim)
        self.norm_v = nn.LayerNorm(embed_dim)
        self.norm_post_attn = nn.LayerNorm(embed_dim)
        self.norm_pre_mlp = nn.LayerNorm(embed_dim)
        self.norm_post_mlp = nn.LayerNorm(embed_dim)
        
        # 4. Learnable Scaling Factors (Gamma)
        # These are initialized to a small value (1e-4) to stabilize early training
        self.gamma1 = nn.Parameter(torch.ones(embed_dim) * 1e-4)
        self.gamma2 = nn.Parameter(torch.ones(embed_dim) * 1e-4)

    def forward(self, x_class, x_particles):
        """
        Args:
            x_class: The class token [Batch, 1, 128]
            x_particles: Particle embeddings from previous blocks [Batch, N, 128]
        """
        
        # --- Stage 1: Attention (Class token queries the particles) ---
        # Pre-Norm for Query, Key, and Value
        # Query comes ONLY from x_class
        q = self.norm_q(x_class)
        # Keys and Values come from the particles
        k = self.norm_k(x_particles)
        v = self.norm_v(x_particles)
        
        # Attention logic: attn(Q, K, V)
        # The output has the same shape as x_class [Batch, 1, 128]
        attn_out, _ = self.attn(q, k, v)
        
        # Post-Norm + Scale + Residual update to x_class
        x_class = x_class + self.gamma1 * self.norm_post_attn(attn_out)
        
        # --- Stage 2: Feed-Forward (MLP) ---
        # Pre-Norm
        x_mlp = self.norm_pre_mlp(x_class)
        
        # MLP and Post-Norm + Scale + Residual update to x_class
        x_class = x_class + self.gamma2 * self.norm_post_mlp(self.mlp(x_mlp))
        
        return x_class

# --- Usage Example within the full ParT Model ---
# Initialize the learnable class token (the "Notebook")
# batch_size = 32, embed_dim = 128
# self.class_token = nn.Parameter(torch.zeros(1, 1, 128))